# Credit Risk Assessment with Explainable AI
**Created:** May 30, 2025

This notebook walks through a project that uses XGBoost to assess credit risk, with SHAP for model explainability. We'll ingest a sample loan dataset, engineer features, train a classifier, evaluate performance, and explain predictions.

In [ ]:
!pip install xgboost shap pandas scikit-learn matplotlib seaborn

## 1. Load and Explore the Data

In [ ]:

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Example dataset from Kaggle: Lending Club or synthetic data
url = "https://raw.githubusercontent.com/selva86/datasets/master/LoanApprovalPrediction.csv"
df = pd.read_csv(url)
df.head()


## 2. Preprocess Data

In [ ]:

# Simple cleaning
df.dropna(inplace=True)
categorical = df.select_dtypes(include='object').columns
df_encoded = pd.get_dummies(df, columns=categorical, drop_first=True)

# Features & target
X = df_encoded.drop("Loan_Status_Y", axis=1)
y = df_encoded["Loan_Status_Y"]


## 3. Train XGBoost Model

In [ ]:

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))


## 4. Explain Predictions using SHAP

In [ ]:

import shap
shap.initjs()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary plot
shap.summary_plot(shap_values, X_test)


## 5. Predict and Explain One Sample

In [ ]:

sample = X_test.iloc[[0]]
pred = model.predict(sample)
print(f"Predicted Credit Risk: {'Approved' if pred[0] else 'Denied'}")

shap.force_plot(explainer.expected_value, shap_values[0], sample)
